## Imports and Environment

In [1]:
import sys
import logging
import os
import gc
import asyncio
import pandas as pd
import torch
from datetime import datetime
import re
from dotenv import load_dotenv
from pypdf import PdfReader
from docx import Document


from llama_index.core import PromptTemplate, QueryBundle, get_response_synthesizer
from llama_index.core.agent.workflow import (
    AgentWorkflow, 
    ToolCall,
    ToolCallResult,
    AgentStream
)
from llama_index.core.tools import FunctionTool
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_index.llms.openai_like import OpenAILike
from llama_index.core import QueryBundle
from llama_index.core.schema import TextNode, NodeWithScore

from deepeval import evaluate
from deepeval.test_case import LLMTestCase, ToolCall
# GEval allows writing in grading rubric in plain english (Pass if... Fail if...) 
from deepeval.metrics import ToolCorrectnessMetric, HallucinationMetric, GEval
from deepeval.test_run import test_run
from deepeval.test_case import LLMTestCaseParams
from deepeval.evaluate import DisplayConfig, AsyncConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Force the code to use the local vLLM server
os.environ["VLLM_API_BASE"] = "http://localhost:8001/v1"

env_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

# Load it
if load_dotenv(env_path):
    print(f"✅ Successfully loaded .env from: {env_path}")
else:
    print(f"❌ Failed to load .env. Check if the file exists at: {env_path}")

# Helper to add paths
def add_pipeline_path(env_var_name):
    path = os.getenv(env_var_name)
    if path and os.path.isdir(path):
        if path not in sys.path:
            sys.path.append(path)
        print(f"Successfully added path for {env_var_name}: {path}")
        return True
    else:
        print(f"Warning: '{env_var_name}' is invalid or not set: {path}")
        return False

# Add paths for all pipelines
add_pipeline_path("VECTOR_PIPELINE_DIR")
add_pipeline_path("SQL_PIPELINE_DIR")
add_pipeline_path("GRAPH_PIPELINE_DIR")

# Dynamic Imports (using module aliases to avoid name collisions)
try:
    import vector_pipeline
    import sql_pipeline
    import graph_pipeline
except ImportError as e:
    print(f"Critical Import Error: {e}")
    print("Ensure all pipeline directories are correctly set in .env")
    sys.exit(1)


vec_pipe = None
sql_pipe = None
graph_pipe = None

✅ Successfully loaded .env from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env
Successfully added path for VECTOR_PIPELINE_DIR: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/Vector Pipeline
Successfully added path for SQL_PIPELINE_DIR: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/SQL Pipeline
Successfully added path for GRAPH_PIPELINE_DIR: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/Graph Pipeline


## Initialise Pipelines

In [2]:
print("Initializing Pipelines...")

# Vector Pipeline
vec_config = vector_pipeline.ConfigManager(env_path=env_path)
vec_models = vector_pipeline.ModelProvider(vec_config)
vec_pipe = vector_pipeline.RAGPipeline(vec_config, vec_models)
raw_docs = vec_pipe.ingestor.ingest_pdfs_from_directory(
    data_directory=vec_pipe.config.data_directory
)
print(f"✅ Ingested {len(raw_docs)} documents.")

# Setup Pipeline with the Ingested Documents
vec_pipe.setup_pipeline(documents=raw_docs) 

# SQL Pipeline
sql_config = sql_pipeline.ConfigManager(env_path=env_path)
db_manager = sql_pipeline.DatabaseManager(sql_config)
engine, tables = db_manager.populate_database()
sql_pipe = sql_pipeline.AdvancedQueryEngine(engine, tables, sql_config)

# Graph Pipeline
graph_config = graph_pipeline.ConfigManager(dotenv_path=env_path)
graph_models = graph_pipeline.ModelRegistry(graph_config)
graph_pipe = graph_pipeline.GraphQueryPipeline(graph_config, graph_models)

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_llm = OpenAI(
    api_key=openai_api_key,
    model="gpt-4o",
    temperature=0.0,
)

print("✅ All Pipelines Ready.")


Initializing Pipelines...
Loaded .env from /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env
Project root set to: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone
Data directory set to: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Vector_Dataset
Connected to vLLM at http://localhost:8001/v1 with model: /models/Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Nanonets OCR model loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/Nanonets-OCR-s
Embedding model loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/intfloat_multilingual_e5_large
Reranker model loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/bge-reranker-v2-m3
Global LlamaIndex settings configured.
Found 8 PDFs in directory.
Skipping ANNUAL CRIME BRIEF 2020.pdf (Already ingested)
Skipping ANNUAL CRIME BRIEF 2021.pdf (Already ingested)
Skipping ANNUAL CRIME BRIEF 2022.pdf (Already ingested)
Skipping Annual Crime Brief 2023.pdf (Already ingested)
Skipping Annual Crime Brief 2024.pdf (Already ingested)
Skipping Annual Scams and Cybercrime Brief 2023.pdf (Already ingested)
Skipping Annual Scams and Cybercrime Brief 2024.pdf (Already ingested)
Skipping Police News Release - Annual Scams and Cybercrime Brief 2022.pdf (Already ingested)
✅ No new files to ingest.
✅ Ingested 

In [3]:
# Release OCR Model from Vector Pipeline to free up GPU memory

print("Cleaning up OCR model...")
del vec_pipe.ingestor.ocr_model
del vec_pipe.ingestor.ocr_processor
del vec_pipe.models.ocr_model
del vec_pipe.models.ocr_processor

gc.collect()
torch.cuda.empty_cache() # <--- Releases VRAM back to OS/PyTorch pool
print(f"GPU Memory Freed. Current: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


Cleaning up OCR model...
GPU Memory Freed. Current: 6.30 GB


## Tools

In [4]:
captured_tool_chunks = []  # Global list to store chunks generated by the vector pipeline tool

def search_spf_data(query: str) -> str:
    """Queries the SPF database."""
    if not query: return "Error: Query cannot be empty."
    clean_query = query.strip().strip('"').strip("'")
    # Get the full response object
    response = vec_pipe.query_engine.query(clean_query)
    
    # # DEBUG: Print the actual retrieved text chunk
    if response.source_nodes:
        formatted_chunks = []
        for node in response.source_nodes:
            # Extract filename from metadata, default to 'Unknown Document' if missing
            file_name = node.node.metadata.get('file_name', 'Unknown Document')
            content = node.node.get_content().strip()
            formatted_chunks.append(f"--- Document: {file_name} ---\n{content}")
            
        return "\n\n".join(formatted_chunks)
    else:
        return "No nodes retrieved."
        
    return str(response)

def search_sps_data(query: str) -> str:
    """Queries the SPS SQL database."""
    if not sql_pipe: return "Error: SPS Pipeline not initialized."
    clean_query = query.strip().strip('"').strip("'")

    # Remove Markdown code blocks (```sql ... ```)
    if "```" in clean_query:
        clean_query = clean_query.replace("``````", "").strip()
    
    # Remove "###" artifacts (e.g., "SELECT ...; ### ANSWER")
    if "###" in clean_query:
        clean_query = clean_query.split("###")[0].strip()

    return str(sql_pipe.query(clean_query))

def search_htx_data(query: str) -> str:
    """Queries the HTX Graph database."""
    if not graph_pipe: return "Error: HTX Pipeline not initialized."
    clean_query = query.strip().strip('"').strip("'")
    return str(graph_pipe.query(clean_query))

# Create FunctionTools
tools = [
    FunctionTool.from_defaults(fn=search_spf_data, name="spf_vector_tool", description="Repository for Singapore public sector policy documentation and regulatory announcements (2020–2026), specifically covering MAS technology risk guidelines, HDB housing schemes, MOM employment criteria, and Parliamentary debates. "
    "Use this tool for queries regarding changing government regulations, eligibility thresholds, and policy implementation timelines. "
    "It contains a mix of consultation papers (drafts), press releases (announcements), and final guidelines. Note that this store deliberately retains historical and superseded versions; ALWAYS compare the 'publication_date' and 'document_status' (e.g., Draft vs. Final) to determine the currently valid rule."),
    FunctionTool.from_defaults(fn=search_sps_data, name="sps_sql_tool", description="Primary database for Singapore Prison Service (SPS) statistical data (2006-2020). "
            "Use this tool for quantitative queries regarding 'Convicted Penal Population' "
            "broken down by 'Year', 'Age Group' (e.g., Below 21, 21-30, 60 Above), and 'Gender'. "
            "It contains structured annual tables suitable for aggregation and trend analysis. "
            "ALWAYS formulate a precise query that specifies the Year, Gender, and Age Group filters immediately "
            "(e.g., 'Total male inmates aged 31-40 in 2015'). "
            "Trust the returned figures as the final official statistics."),
    FunctionTool.from_defaults(fn=search_htx_data, name="htx_graph_tool", description="Primary database for Home Team Science & Technology Agency (HTX) knowledge, built as a knowledge graph from the FY2023 Annual Report." 
            "Use this tool for queries about science and technology capabilities, operational projects, and innovation initiatives across the Home Team" 
            "(e.g., robotics, biometrics, cybersecurity, CBRNE, XR/VR training systems)." 
            "It is best suited for questions on specific HTX projects (such as Rover-X, Marine Video Analytics for rescue, autonomous robots, or deepfake detection), strategic partnerships, and how technologies are deployed to support SPF, SCDF, ICA, SPS, and other Home Team departments. ALWAYS phrase queries as concrete questions about a particular capability, project, or domain (e.g., 'What technologies does HTX use to counter hostile drones?').")
]


## Agent

In [ ]:
agent_model_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/Llama-3.1-8B-Instruct"

agent_llm = HuggingFaceLLM(
    model_name=agent_model_path,
    tokenizer_name=agent_model_path,
    context_window=32000, 
    max_new_tokens=4096,
    generate_kwargs={
        "temperature": 0.1, 
        "do_sample": True,
        "repetition_penalty": 1.15, # Mild penalty to reduce exact repetition
        "pad_token_id": 128001, # 128001 is the ID for <|end_of_text|> token which LLama 3 uses for padding
        "eos_token_id": [128001, 128009] # 129008 is the ID for <|eot_id|>, End of Turn, emits when finish answering a user's prompt
    },
    model_kwargs={"dtype": torch.bfloat16},
    device_map="auto",
)

# Get current date (e.g., "Tuesday, January 20, 2026")
current_date = datetime.now()
current_date_str = current_date.strftime("%A, %B %d, %Y")
current_year = current_date.year
last_year = current_year - 1

print(f"✅ Loaded new Agent LLM: {agent_model_path}")

# Synthesis Prompt Template for final answer generation after tool calls
synthesis_prompt = PromptTemplate(
    "You are an expert analyst drafting a formal response based on Singapore government official documents.\n"
    "Use the context ONLY; do not use any external knowledge.\n\n"

    "### TASK\n"
    "1. Carefully examine the context for conflicting or missing facts about the question. A 'conflict' occurs if:\n"
    "   - Source A explicitly contradicts Source B.\n"
    "   - Source A contains the answer, but Source B does not mention the answer at all.\n"
    "2. Think through whether both sources agree and provide the exact same answer.\n"
    "3. Decide if you should ANSWER or REFUSE.\n\n"

    "### FORMAT INSTRUCTIONS\n"
    "Do NOT output your thought process, reasoning or XML tags.\n"
    "If there is a conflict OR if any source is missing the answer, write EXACTLY:\n"
    "\"I am unable to answer this question due to conflicting contexts found in [Source A] and [Source B].\"\n"
    "Otherwise, provide a direct and concise answer in one paragraph, citing the specific source(s).\n"
    "</answer>\n\n"

    "### CONTEXT\n"
    "{context_str}\n\n"

    "### QUESTION\n"
    "{query_str}\n\n"

    "### ANSWER:\n"
)

# Initialize the synthesizer with your specific LLM (e.g., agent_llm or openai_llm)
synthesizer = get_response_synthesizer(
    llm=agent_llm, # Or openai_llm if you prefer GPT-4o for synthesis
    text_qa_template=synthesis_prompt
)

# Initialize Workflow
workflow = AgentWorkflow.from_tools_or_functions(tools, llm=agent_llm, system_prompt="" \
        "### CONTEXT (C) \n"
        "You are the Chief Data Orchestrator for the Singapore Home Team Agentic RAG pipeline. "
        "You manage access to three distinct databases:\n"
        "1. **SPF Vector Store**: Unstructured reports on crime and scams (Singapore Police Force).\n\n"
        # # For inconsistent data evaluation only
        # "1. **SPF Vector Store**: Repository for Singapore public sector policy documentation and regulatory announcements (2020–2026), specifically covering MAS technology risk guidelines, HDB housing schemes, MOM employment criteria, and Parliamentary debates.\n\n"
        "2. **SPS SQL Database**: Structured demographic data on convicted penal populations (Singapore Prison Service).\n\n"
        "3. **HTX Graph Database**: Knowledge graph on science, technology, and innovation projects (Home Team Science & Tech Agency).\n\n"
        "4. **Context Window**: Your available data sources cover up to {current_year}.\n"
        "5. **Temporal Context**: Today is **{current_date_str}**.\n"
        "   - **CRITICAL**: When the user asks for generic stats (e.g., 'How many keys?'), "
        "     you MUST explicitly append the current reporting year to the query.\n"
        "   - Query Format: '{topic} {current_year} {last_year}'\n"
        "   - If you cannot find relevant information after 3 tool calls, respond: 'I cannot find this information in the available documents.' Do NOT invent answers or hallucinate.\n" 
        "   **RULE FOR 'LATEST' DATA**:\n"
        "   - Always prioritize the **Current Year ({current_year})**.\n"
        "   - To ensure robust retrieval (in case {current_year} data is sparse), you must ALSO include the **Previous Year ({last_year})** as a secondary keyword.\n"
        "   - Query format: '{topic} {current_year} {last_year} ...'\n\n"


        "### OBJECTIVE (O) \n"
        "1. Analyze the user's input for keywords relating to specific agencies (SPF, SPS, HTX) or topics (Crime, Prisoners, Technology).\n"
        "2. Select the correct tool (`spf_vector_tool`, `sps_sql_tool`, or `htx_graph_tool`).\n"
        "3. **QUERY REFINEMENT**: \n"
        "   - If NO date is specified, AUTOMATICALLY append keywords: '{current_year} {last_year} latest statistics'.\n"
        "   - This ensures the vector store retrieves the most recent documents.\n"
        "4. Call that tool with the exact query.\n"
        "5. STOP immediately. Do not analyze the output.\n\n"
        "6. Output ONLY the tool call with the REFINED query.\n\n"

        "### EXAMPLES \n"
        "- User: 'What is the flat classification system?' (No date)\n"
        "  Tool Call: spf_vector_tool(query='flat classification system 2025')\n"
        "- User: 'How many inmates in 2020?' (Date present)\n"
        "  Tool Call: sps_sql_tool(query='How many inmates in 2020?')\n\n"
        

        "### STYLE (S) \n"
        "Decisive, precise, and classification-focused.\n"

        "### TONE (T) \n"
        "Objective and neutral.\n"

        "### AUDIENCE (A) \n"
        "A Python runtime environment waiting for a tool call.\n\n"

        "### RESPONSE (R) \n"
        "Output ONLY the tool call and tool output.\n"
        "Do NOT attempt to answer the question yourself.\n"
        "If the query mixes topics (e.g., \"Tech used by Prisons\"), prioritize the agency responsible for the *subject* of the query (e.g., if asking about the *tech*, route to HTX; if asking about the *inmates*, route to SPS).\n"
        )

async def run_agent_for_eval(user_query: str):
    """
    Wrapper to run the agent and capture tool usage + retrieved chunks for DeepEval.
    """
    handler = workflow.run(
        user_msg=user_query,
        timeout=30,  # Set a timeout for the entire workflow run (adjust as needed)
        max_iterations=3, # Stop the agent after 3 tool calls to prevent infinite loops and ensure timely responses for evaluation purposes
        early_stopping_method="generate" # Generate a final answer instead of throwing an error
        )
    
    # final_response = ""
    tool_logs = []
    # retrieved_chunks = []  
    tool_observations = [] # Capture text returned by tools
    
    # Iterate through ALL events
    async for event in handler.stream_events():
        
        # Capture Tool INPUT
        if isinstance(event, ToolCall):
            log_entry = f"Tool Call: {event.tool_name}\nInput: {event.tool_kwargs}"
            tool_logs.append(log_entry)
            
        # Capture Tool OUTPUT
        elif isinstance(event, ToolCallResult):
            output_text = str(event.tool_output)

            # Controller to stop if getting repeated empty results
            if output_text == "[]" or "No nodes retrieved" in output_text:
                empty_results_count += 1
                if empty_results_count >= 2:
                    print("Controller intervention: 2 empty tool results. Aborting further tool calls.")
                    # We break out of the stream early to save tokens and time
                    handler.cancel() 
                    break 
            
            log_entry = f"Tool Result ({event.tool_name}):\nOutput: {output_text[:500]}..."
            tool_logs.append(log_entry)
            tool_observations.append({
                "tool": event.tool_name,
                "text": output_text
            })

    # Chunk Filtering and Answer Generation
    retrieved_chunks = []
    nodes_for_synthesis = []

    # Cleaning Function for Final Response
    def clean_llm_output(raw_text: str) -> str:
        """Cleans up hallucinated tags, sources, and infinite loops from the LLM."""
        if not isinstance(raw_text, str):
            return str(raw_text)
            
        text = raw_text.strip()
        
        # Remove XML/HTML tags like <answer>, </answer>, <think>
        text = re.sub(r'</?[a-zA-Z]+>', '', text, flags=re.IGNORECASE)
        
        # Remove ### SOURCE / #Source sections
        # If the LLM generated a "### RESPONSE", extract only want what comes after it
        if re.search(r'#+\s*RESPONSE', text, flags=re.IGNORECASE):
            # Split by the Response header and take the last part
            text = re.split(r'#+\s*RESPONSE:?', text, flags=re.IGNORECASE)[-1]
            
        # If it generated a "### SOURCE" at the end, extract what came before it.
        if re.search(r'#+\s*SOURCE', text, flags=re.IGNORECASE):
            # Split by the Source header and take the first part
            text = re.split(r'#+\s*SOURCE', text, flags=re.IGNORECASE)[0]

        # Remove other internal thought process headers
        text = re.sub(r'(?i)#?answer:?', '', text)
        text = re.sub(r'(?i)#?question:?', '', text)
        text = re.sub(r'(?i)revised answer:?', '', text)
        
        # Replace placeholder source brackets with "conflicting information"
        if re.search(r'\[Source [A-Z]\]|\[Context \d+\]', text, flags=re.IGNORECASE):
            if "unable to answer" in text.lower() or "conflicting" in text.lower():
                return "I am unable to answer this question due to conflicting information in the documents."
                
        # Handle the "infinite 'answer'" loop
        text = re.sub(r'(?i)(\banswer\b\s*){4,}', '', text)
        
        # Handle the "I am unable to answer... [Source A]" infinite loop 
        match = re.search(r'(I am unable to answer this question due to conflicting[^.]*\.)', text)
        if match:
            return match.group(1).strip()
            
        # Final cleanup of extra whitespace and loose periods left behind
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    # Convert tool observations into LlamaIndex Nodes
    for i, obs in enumerate(tool_observations):
        # Store the output text as a chunk of context
        chunk_text = obs["text"]
        
        # Package for evaluation tracking
        retrieved_chunks.append({
            "chunk_id": i + 1,
            "text": chunk_text,
            "source_file": obs["tool"], # Identify which tool provided this chunk
            "score": 1.0 
        })
        
        # Package into LlamaIndex Node format for the synthesizer
        node = TextNode(text=chunk_text)
        nodes_for_synthesis.append(NodeWithScore(node=node, score=1.0))

    # Generate final answer using the second prompt template
    if nodes_for_synthesis:
        response_obj = synthesizer.synthesize(
            query=user_query,
            nodes=nodes_for_synthesis
        )
        
        # Get raw string
        raw_response = str(response_obj)

        # Clean raw response
        final_response = clean_llm_output(raw_response)
    else:
        # Fallback if the agent decided to call zero tools
        final_response = "No context was retrieved. I cannot find this information."
    
    return final_response, tool_logs, retrieved_chunks, tool_observations




Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Loaded new Agent LLM: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/Llama-3.1-8B-Instruct


## Evaluation

In [ ]:
project_root = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone"
dataset_rel_path = os.getenv("AGENT_BENCHMARK_DATASET_DIR") 
full_path = os.path.join(project_root, dataset_rel_path)
print(f"Loading from: {full_path}")
df = pd.read_csv(full_path).head(5)

# Define Metrics
# ToolCorrectness: Checks if the right tool was picked
# Purely math-based metric
# number of correctly selected tools divide by total number of expected tools
# if strict_mode = True, then tools must be called in the exact expected order
# otherwise, strict_mode = False by default
tool_metric = ToolCorrectnessMetric(threshold=0.5)

# Define Task Completion using G-Eval
# Calculated using Weighted Log-Probability Method (if using GPT-4)
# Direct Scoring method (if using other LLMs)
# Rely on semantic understanding of LLM to judge if the final answer satisfies user intent
task_completion_metric = GEval(
    name="Task Completion",
    criteria="Determine if the agent has fully satisfied the user's intent. Pass if the final answer directly addresses the prompt. Fail if the agent says 'I cannot answer' or asks for clarification unnecessarily.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o",
    threshold=0.5
)

test_cases = []

print(f"🚀 Starting Evaluation on {len(df)} rows...")

# Loop through questions and generate Test Cases
for index, row in df.iterrows():
    query = row['question']
    expected_tool_name = row['tool']
    
    # Handle NaNs: If 'tool' is empty in CSV
    if pd.isna(expected_tool_name):
        expected_tools = [] 
    else:
        expected_tools = [ToolCall(name=expected_tool_name)]

    print(f"[{index+1}/{len(df)}] Testing: {query[:50]}...")

    actual_output, tool_logs, retrieved_chunks, tool_observations = await run_agent_for_eval(query)

    # Extract the list of tool names from the agent's observations
    actual_tool_names = [obs["tool"] for obs in tool_observations]

    # Filter out 'None', empty strings, and 'none' (case-insensitive)
    clean_tool_names = [
        name for name in actual_tool_names 
        if name and str(name).strip().lower() != "none"
    ]

    # Remove duplicates while preserving the order they were called
    clean_tool_names = list(dict.fromkeys(clean_tool_names))

    # Convert string list to DeepEval's ToolCall objects
    actual_tools = [ToolCall(name=name) for name in clean_tool_names]

    # --- CREATE TEST CASE ---
    test_case = LLMTestCase(
        input=query,
        actual_output=str(actual_output),
        tools_called=actual_tools,
        expected_tools=expected_tools
    )
    test_cases.append(test_case)

def extract_tools_from_log(log_text):
    if not isinstance(log_text, str): return []
    # Find names inside ToolCall(name="...")
    names = re.findall(r'name=["\'](.*?)["\']', log_text)
    return list(dict.fromkeys(names))

print("✅ Generation Complete. Running DeepEval...")

# Run Evaluation
results = evaluate(
    test_cases=test_cases, 
    metrics=[tool_metric, task_completion_metric],
)

results_data = []

# Access the list inside the wrapper
actual_results_list = results.test_results if hasattr(results, 'test_results') else results

for i, r in enumerate(actual_results_list):
    # Initialize Row with basic info
    row = {
        "Input": getattr(r, "input", "N/A"),
        "Actual Output": getattr(r, "actual_output", "N/A"),
        "Passed": getattr(r, "success", False),
        "Tools Called": [],   # Default empty
        "Expected Tools": []  # Default empty
    }

    # --- Extract Metrics & Recover Missing Tool Data ---
    metrics_list = getattr(r, "metrics", getattr(r, "metrics_data", []))
    
    for metric in metrics_list:
        name = getattr(metric, "name", "Unknown")
        score = getattr(metric, "score", 0.0)
        reason = getattr(metric, "reason", "")
        
        row[name] = score
        row[f"{name} Reason"] = reason
        
        if name == "Tool Correctness":
            verbose_logs = getattr(metric, "verbose_logs", "")
            
            # Extract "Tools Called" section from logs
            if "Tools Called:" in verbose_logs:
                # Split log to isolate the "Tools Called" part
                parts = verbose_logs.split("Tools Called:")
                if len(parts) > 1:
                    called_part = parts[1].split("Available Tools:")[0] # Stop before next section
                    row["Tools Called"] = extract_tools_from_log(called_part)
            
            # Extract "Expected Tools" section
            if "Expected Tools:" in verbose_logs:
                parts = verbose_logs.split("Expected Tools:")
                if len(parts) > 1:
                    exp_part = parts[1].split("Tools Called:")[0]
                    row["Expected Tools"] = extract_tools_from_log(exp_part)

    results_data.append(row)

# Save & Sort
df_final = pd.DataFrame(results_data)
# Sort by Input to match benchmark order (roughly)
df_final = df_final.sort_values(by="Input") 
df_final.to_csv("(test)tool_calling_results.csv", index=False)

print(f"✅ Evaluation Complete. Saved {len(df_final)} rows to (test)tool_calling_results.csv")

Loading from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Benchmark Dataset/agent_benchmark.csv
🚀 Starting Evaluation on 5 rows...
[1/5] Testing: What does the acronym SPF stand for in the context...
[2/5] Testing: What is the chemical symbol for a diamond?...
[3/5] Testing: What is the definition of a "statutory board"? ...
[4/5] Testing: How many years are in a standard decade?...
[5/5] Testing: What is the primary function of a "fire station"? ...

=== RAW LLM OUTPUT ===
MATCH (c:Concept {name: "C4I"})-[:HAS_DESCRIPTION]->(d:Description) 
RETURN d.name AS definition


Generated and Cleaned Cypher Query:
MATCH (c:Concept {name: "C4I"})-[:HAS_DESCRIPTION]->(d:Description) 
RETURN d.name AS definition

Formatted output for LLM:
Found 1 results. definition: Command, Control, Communications, Computers and Intelligence

Query Results:
Result 1: definition=Command, Control, Communications, Computers and Intelligence

✅ Generation Complete. Running DeepEval

✨ You're running DeepEval's latest Tool Correctness Metric! (using None, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Task Completion [GEval] Metric! (using gpt-4o, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Tool Correctness (score: 1.0, threshold: 0.5, strict: False, evaluation model: None, reason: [
	 Tool Calling Reason: All expected tools [] were called (order not considered).
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]
, error: None)
  - ❌ Task Completion [GEval] (score: 0.06689271593697602, threshold: 0.5, strict: False, evaluation model: gpt-4o, reason: The response fails to address the user's intent, which is to know the number of years in a standard decade. It does not provide a complete or satisfactory answer, as it states 'I cannot find this information,' which is unnecessary and indicates a lack of alignment with the user's expectations., error: None)

For test case:

  - input: How many years are in a standard decade?
  - actual output: No context was retrieved. I cannot find this information.
  - expected output: None
  - context: None
  - retrieval context: None


Metrics Summary

  - ❌ Tool Correctne

⚠ WARNING: No hyperparameters logged.
» ]8;id=925640;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 7.52s | token cost: 0.014965000000000001 USD)
» Test Results (5 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 5

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

✅ Evaluation Complete. Saved 5 rows to (test)tool_calling_results.csv


## Reframed Inconsistent Data Evaluation

In [ ]:
correctness_metric = GEval(
    name="Ground Truth Correctness",
    criteria="Determine if the actual output is factually correct and semantically equivalent to the expected output.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model="gpt-4o",
    threshold=0.5
)

conflict_prompt = PromptTemplate(
    "DO NOT act as a general conversational assistant. DO NOT use any external knowledge, assumptions, or prior training data outside of the provided context. DO NOT draft informal responses.\n\n"

    "### RESTRICTIONS\n"
    "1. DO NOT ignore missing information. If one source mentions a fact but another source is entirely silent on it, DO NOT treat them as agreeing.\n"
    "2. DO NOT provide an answer if Source A contradicts Source B in any way.\n"
    "3. DO NOT attempt to guess, synthesize a compromise, or choose one source over another if they do not perfectly align.\n\n"

    "### FORMAT PENALTIES\n"
    "DO NOT output any XML tags. \n"
    "DO NOT skip listing statements from the sources.\n"
    "DO NOT skip checking for disagreements or missing data.\n"
    "DO NOT make a final decision without full agreement across all sources.\n"

    "### ANSWER CONSTRAINTS\n"
    "If a conflict exists, or if any source is missing the answer, DO NOT write anything other than exactly:\n"
    "\"I am unable to answer this question due to conflicting contexts found in [Source A] and [Source B].\"\n"
    "If there is absolutely no conflict, DO NOT write multiple paragraphs, and DO NOT forget to cite the specific source(s) used.\n\n"

    "### CONTEXT\n"
    "{context_str}\n\n"

    "### QUESTION\n"
    "{query_str}\n\n"
)


# This uses the LLM from models provider and the prompt above
synthesizer = get_response_synthesizer(
    llm=vec_models.llm,
    text_qa_template=conflict_prompt
)

# Load Data & Run
project_root = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone"
dataset_rel_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Benchmark Dataset/(small)inconsistent_benchmark.csv"
full_path = os.path.join(project_root, dataset_rel_path)
df_benchmark = pd.read_csv(full_path)

results = []
test_cases = [] # Store test cases for DeepEval

print(f"🚀 Testing Conflict Prompt on {len(df_benchmark)} rows...")

for index, row in df_benchmark.iterrows():
    question = row['question']
    ground_truth_context = row['context']
    category = row['category'] # 'conflict' or 'non-conflict'
    ground_truth_answer = row['gt_answer']
    
    # Create a Node with the CSV context (simulates perfect retrieval)
    forced_nodes = [
        NodeWithScore(node=TextNode(text=ground_truth_context), score=1.0)
    ]
    
    try:
        # Generate Response using the pipeline's logic
        response_obj = synthesizer.synthesize(
            query=QueryBundle(question),
            nodes=forced_nodes
        )
        response_text = str(response_obj).strip()
    except Exception as e:
        response_text = f"Error: {str(e)}"
    
    # Check for refusal keywords defined in the prompt
    # e.g., "unable to answer", "conflicting contexts"
    is_refusal = ("unable to answer" in response_text.lower() and "conflicting" in response_text.lower())
    results.append({
        "question": question,
        "context": ground_truth_context,
        "category": category,
        "is_refusal": is_refusal,
        "actual_output": response_text
    })

    # DeepEval Test Case with expected_output
    test_case = LLMTestCase(
        input=question,
        actual_output=response_text,
        expected_output=ground_truth_answer,
        context=[ground_truth_context]
    )

    test_cases.append(test_case)

evaluation_results = evaluate(
    test_cases=test_cases,
    metrics=[correctness_metric],
)

# Extract scores and reasons back into results dictionary
for i, test_result in enumerate(evaluation_results.test_results):
    try:
        metric_data = test_result.metrics_data[0]  # Get the first metric
        results[i]["geval_score"] = metric_data.score
        results[i]["geval_reason"] = metric_data.reason
    except Exception as e:
        results[i]["geval_score"] = 0.0
        results[i]["geval_reason"] = f"Error extracting score: {str(e)}"

# ---------------------------------------------------------
# CALCULATE METRICS
# ---------------------------------------------------------
results_df = pd.DataFrame(results)

# Map categories to binary values
# conflict = 1 (Positive Class), non-conflict = 0 (Negative Class)
y_true = results_df['category'].map({'conflict': 1, 'non-conflict': 0})
y_pred = results_df['is_refusal'].astype(int)

# Calculate Scores
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

tn, fp, fn, tp = cm.ravel()

print("\n### 📊 Conflict Detection Metrics ###")
print(f"Accuracy:  {accuracy:.2%}")
print(f"Precision: {precision:.2%} (Reliability: When it refuses, is it right?)")
print(f"Recall:    {recall:.2%}    (Safety: Did it catch all conflicts?)")
print(f"F1 Score:  {f1:.2%}")
print("-" * 30)
print(f"✅ True Positives (Correct Refusals): {tp}")
print(f"❌ False Positives (Unnecessary Refusals): {fp}")
print(f"⚠️ False Negatives (Missed Conflicts): {fn}")
print(f"👍 True Negatives (Correct Answers): {tn}")

# Save detailed results
results_df.to_csv("(3neg_prompt)conflict_test_results.csv", index=False)
print("\n✅ Detailed results saved to '(3neg_prompt)conflict_test_results.csv'")
